In [ ]:
# ============================================
# COLAB 1: ETL - TITANIC (DESDE DRIVE)
# ============================================

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

# 1. CARGAR DESDE TU DRIVE
print("📥 Cargando dataset desde Drive...")
df = pd.read_csv('/content/drive/MyDrive/titanicBigdata/titanic.csv')

print(f"✅ Dataset cargado: {len(df)} registros")
print(f"Columnas: {df.columns.tolist()}")
print(f"Nulos:\n{df.isnull().sum()}")

# 2. EJECUTAR ETL
print("\n🔄 Ejecutando ETL...")

# 2.1 Limpieza
df_clean = df.drop(['PassengerId', 'Ticket', 'Cabin', 'Name'], axis=1)
df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].mean())
df_clean['Embarked'] = df_clean['Embarked'].fillna('S')

# 2.2 Variables derivadas
df_clean['AgeGroup'] = pd.cut(
    df_clean['Age'],
    bins=[0, 18, 30, 50, 100],
    labels=['Child', 'Young_Adult', 'Adult', 'Senior']
)

df_clean['FamilySize'] = df_clean['SibSp'] + df_clean['Parch'] + 1
df_clean['IsAlone'] = (df_clean['FamilySize'] == 1).astype(int)

# 2.3 Codificación para ML
df_clean['Sex_Code'] = df_clean['Sex'].map({'male': 0, 'female': 1})
df_clean['Embarked_Code'] = df_clean['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
df_clean['AgeGroup_Code'] = df_clean['AgeGroup'].map({
    'Child': 0, 'Young_Adult': 1, 'Adult': 2, 'Senior': 3
})

# 3. GUARDAR EN DRIVE (¡AQUÍ ESTÁN TUS ARCHIVOS LIMPIOS!)

# 3.1 Para Neo4j - con todas las columnas
df_clean.to_csv('/content/drive/MyDrive/titanicBigdata/titanic_for_neo4j.csv', index=False)
print("✅ Guardado: titanic_for_neo4j.csv")

# 3.2 Para ML - solo numéricas
df_ml = df_clean[['Pclass', 'Sex_Code', 'Age', 'SibSp', 'Parch', 'Fare',
                  'Embarked_Code', 'AgeGroup_Code', 'FamilySize', 'IsAlone', 'Survived']]
df_ml.to_csv('/content/drive/MyDrive/titanicBigdata/titanic_for_ml.csv', index=False)
print("✅ Guardado: titanic_for_ml.csv")

# 3.3 Dataset completo limpio
df_clean.to_csv('/content/drive/MyDrive/titanicBigdata/titanic_clean.csv', index=False)
print("✅ Guardado: titanic_clean.csv")

print("\n✅ ETL COMPLETADO - Archivos en Drive:")
print("  • titanic_for_neo4j.csv (para grafos)")
print("  • titanic_for_ml.csv (para ML)")
print("  • titanic_clean.csv (completo)")

# 4. VERIFICAR
print(f"\n📊 Resumen datos limpios:")
print(f"• Total: {len(df_clean)} registros")
print(f"• Supervivencia: {df_clean['Survived'].mean():.2%}")
print(f"• Edad promedio: {df_clean['Age'].mean():.1f}")
print(f"• Grupos de edad: {df_clean['AgeGroup'].value_counts().to_dict()}")

Mounted at /content/drive
📥 Cargando dataset desde Drive...
✅ Dataset cargado: 891 registros
Columnas: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']
Nulos:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

🔄 Ejecutando ETL...
✅ Guardado: titanic_for_neo4j.csv
✅ Guardado: titanic_for_ml.csv
✅ Guardado: titanic_clean.csv

✅ ETL COMPLETADO - Archivos en Drive:
  • titanic_for_neo4j.csv (para grafos)
  • titanic_for_ml.csv (para ML)
  • titanic_clean.csv (completo)

📊 Resumen datos limpios:
• Total: 891 registros
• Supervivencia: 38.38%
• Edad promedio: 29.7
• Grupos de edad: {'Young_Adult': 447, 'Adult': 241, 'Child': 139, 'Senior': 64}


In [2]:
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

# ============================================
# COLAB 1: ETL - TITANIC (CORREGIDO)
# ============================================


# 1. CARGAR DATOS
print("📥 Cargando dataset desde Drive...")
df = pd.read_csv('/content/drive/MyDrive/titanicBigdata/titanic.csv')

print(f"✅ Dataset cargado: {len(df)} registros")

# 2. EJECUTAR ETL
print("\n🔄 Ejecutando ETL...")

# 2.1 Limpieza
df_clean = df.drop(['PassengerId', 'Ticket', 'Cabin', 'Name'], axis=1)
df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].mean())
df_clean['Embarked'] = df_clean['Embarked'].fillna('S')

# 2.2 Variables derivadas (¡PARA TODOS!)
df_clean['AgeGroup'] = pd.cut(
    df_clean['Age'],
    bins=[0, 18, 30, 50, 100],
    labels=['Child', 'Young_Adult', 'Adult', 'Senior']
)

df_clean['FamilySize'] = df_clean['SibSp'] + df_clean['Parch'] + 1
df_clean['IsAlone'] = (df_clean['FamilySize'] == 1).astype(int)

# 2.3 Codificación para ML
df_clean['Sex_Code'] = df_clean['Sex'].map({'male': 0, 'female': 1})
df_clean['Embarked_Code'] = df_clean['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
df_clean['AgeGroup_Code'] = df_clean['AgeGroup'].map(
    {'Child': 0, 'Young_Adult': 1, 'Adult': 2, 'Senior': 3}
)

# ============================================
# 3. GUARDAR 3 ARCHIVOS DIFERENTES
# ============================================

# 3.1 Para NEO4J - TODAS las columnas (incluye nombres y categóricas)
#     ¡Este es el que usas en Neo4j!
df_neo4j = df_clean.copy()
df_neo4j.to_csv('/content/drive/MyDrive/titanicBigdata/titanic_for_neo4j.csv', index=False)
print("✅ titanic_for_neo4j.csv - 14 columnas (todo)")

# 3.2 Para ML - SOLO NUMÉRICAS (lo que necesita el modelo)
#     ¡Este es el que usas en Scikit-learn!
df_ml = df_clean[['Pclass', 'Sex_Code', 'Age', 'SibSp', 'Parch', 'Fare',
                  'Embarked_Code', 'AgeGroup_Code', 'FamilySize', 'IsAlone', 'Survived']]
df_ml.to_csv('/content/drive/MyDrive/titanicBigdata/titanic_for_ml.csv', index=False)
print("✅ titanic_for_ml.csv - 11 columnas (solo números)")

# 3.3 Para POWER BI - Columnas con NOMBRES AMIGABLES
#     ¡Este es el que usas en Power BI para visualizar!
df_powerbi = df_clean.copy()
# Renombrar columnas para que sean más amigables en Power BI
df_powerbi = df_powerbi.rename(columns={
    'Pclass': 'Clase',
    'Sex': 'Genero',
    'Age': 'Edad',
    'SibSp': 'Hermanos_Conyuge',
    'Parch': 'Padres_Hijos',
    'Fare': 'Tarifa',
    'Embarked': 'Embarcacion',
    'Survived': 'Sobrevivio',
    'AgeGroup': 'Grupo_Edad',
    'FamilySize': 'Tamano_Familia',
    'IsAlone': 'Viaja_Solo',
    'Sex_Code': 'Codigo_Genero',
    'Embarked_Code': 'Codigo_Embarcacion',
    'AgeGroup_Code': 'Codigo_GrupoEdad'
})
df_powerbi.to_csv('/content/drive/MyDrive/titanicBigdata/titanic_clean.csv', index=False)
print("✅ titanic_clean.csv - 14 columnas (nombres amigables para Power BI)")

print("\n✅ ETL COMPLETADO - 3 ARCHIVOS DIFERENTES GUARDADOS:")

# ============================================
# 4. VERIFICAR DIFERENCIAS
# ============================================

print("\n📊 COMPARATIVO DE ARCHIVOS:")

print(f"\n1. titanic_for_neo4j.csv ({len(df_neo4j.columns)} columnas):")
print(f"   Columnas: {df_neo4j.columns.tolist()}")
# 'Name' column was dropped, so it cannot be accessed here.
print(f"   Primer registro: {df_neo4j.iloc[0][['Sex', 'Age', 'Survived']].to_dict()}")

print(f"\n2. titanic_for_ml.csv ({len(df_ml.columns)} columnas):")
print(f"   Columnas: {df_ml.columns.tolist()}")
print(f"   Primer registro: {df_ml.iloc[0].to_dict()}")

print(f"\n3. titanic_clean.csv ({len(df_powerbi.columns)} columnas):")
print(f"   Columnas: {df_powerbi.columns.tolist()}")
# 'Nombre' column (renamed from 'Name') was dropped, so it cannot be accessed here.
print(f"   Primer registro: {df_powerbi.iloc[0][['Genero', 'Edad', 'Sobrevivio']].to_dict()}")

print("\n🎉 ETL COMPLETADO CORRECTAMENTE")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📥 Cargando dataset desde Drive...
✅ Dataset cargado: 891 registros

🔄 Ejecutando ETL...
✅ titanic_for_neo4j.csv - 14 columnas (todo)
✅ titanic_for_ml.csv - 11 columnas (solo números)
✅ titanic_clean.csv - 14 columnas (nombres amigables para Power BI)

✅ ETL COMPLETADO - 3 ARCHIVOS DIFERENTES GUARDADOS:

📊 COMPARATIVO DE ARCHIVOS:

1. titanic_for_neo4j.csv (14 columnas):
   Columnas: ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'AgeGroup', 'FamilySize', 'IsAlone', 'Sex_Code', 'Embarked_Code', 'AgeGroup_Code']
   Primer registro: {'Sex': 'male', 'Age': 22.0, 'Survived': 0}

2. titanic_for_ml.csv (11 columnas):
   Columnas: ['Pclass', 'Sex_Code', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked_Code', 'AgeGroup_Code', 'FamilySize', 'IsAlone', 'Survived']
   Primer registro: {'Pclass': 3.0, 'Sex_Code': 0.0, 'Age': 22.0, 'SibSp': 1.0, 'Parc